In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
data_path_hsa = '../../data/raw/HSA/'  # specify your path here
folder = Path(data_path_hsa)  # change to your folder path
print(f'Reading CSV files from {folder}')
csv_files = sorted(folder.glob('*.csv'))

if not csv_files:
    print(f'No CSV files found in {folder}')
else:
    # read each CSV into a dict of DataFrames keyed by filename (without suffix)
    dfs = {p.stem: pd.read_csv(p) for p in csv_files}

    # optional: concatenate all into a single DataFrame
    combined = pd.concat(dfs.values(), ignore_index=True)

    print(f'Read {len(dfs)} files. Combined shape: {combined.shape}')

Reading CSV files from ..\..\..\data\raw\HSA
No CSV files found in ..\..\..\data\raw\HSA


In [ ]:
def make_continuous(x, threshold=None):
    """
    Unwraps a resetting position signal.
    x: array-like
    threshold: negative jump threshold (auto if None)
    """
    x = np.array(x)
    dx = np.diff(x)

    # Auto threshold: any large negative jump
    if threshold is None:
        threshold = -0.5 * np.max(x)  

    offset = 0
    x_cont = np.zeros_like(x)

    for i in range(len(x)):
        if i > 0 and dx[i-1] < threshold:
            # Counter reset detected
            offset += x[i-1]
        x_cont[i] = x[i] + offset

    return x_cont


In [17]:
# rename columns in all loaded DataFrames and save renamed CSVs to a new folder
prepro_path = '../../data/preprocessed/HSA' 
#prepro_path.mkdir(parents=True, exist_ok=True)

# rename columns for clarity
col_map = {
    '+/Nck/!SD/nckServoDataActCurr32 [u1; 4]': 'iqAx4',
    '+/Channel/!RP/rpa [u1; 11]': 'rpa',
    '+/Nck/!SD/nckServoDataActPower32 [u1; 4]': 'powerAx4',
    '+/Nck/!SD/nckServoDataActVelMot32 [u1; 4]': 'velAx4'
}


for stem, df in list(dfs.items()):
    # apply the column mapping (rename ignores keys not present)
    df_renamed = df.rename(columns=col_map)
    # update the in-memory dict to hold renamed DataFrame
    dfs[stem] = df_renamed
    # save with the same filename (stem) into the new folder
    df_renamed = df_renamed.drop(columns="rpa")
    out_file = f'{prepro_path}/{stem}.csv'
    df_renamed.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")

NameError: name 'dfs' is not defined

In [ ]:
# Get the first file from dfs
first_file = list(dfs.values())[55]

# Plot velAx4 column
first_file.plot(x='time', y='velAx4', figsize=(12, 4), title='velAx4 - First File')

In [ ]:
# Find the maximum velAx4 value across all files
max_vel = -np.inf
max_files = []

for stem, df in dfs.items():
    file_max = df['velAx4'].max()
    file_max_idx = df['velAx4'].idxmax()
    
    if file_max > max_vel:
        max_vel = file_max
        max_files = [(stem, file_max_idx, file_max)]
    elif file_max == max_vel:
        max_files.append((stem, file_max_idx, file_max))

# Print results
print(f"Maximum velAx4 value: {max_vel}\n")
for stem, idx, val in max_files:
    print(f"File: {stem}, Index: {idx}, Value: {val}")

In [ ]:
dd =  542034.0/10
print(dd)